# 00_config · Configuración del entorno
Crea (si no existen) el esquema del proyecto, el volumen `raw` y una carpeta por fuente.
Define las variables que usan el resto de notebooks mediante `%run ./00_config`.
Idempotente: se puede ejecutar tantas veces como se quiera.

In [0]:
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("esquema", "proyecto_etl", "Esquema")

CATALOGO = dbutils.widgets.get("catalogo").strip().lower()
ESQUEMA = dbutils.widgets.get("esquema").strip().lower()
VOLUMEN = "raw"
FUENTES = ["bicimad", "calidad_aire", "openmeteo"]

RAW = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
RUTAS = {f: f"{RAW}/{f}" for f in FUENTES}

In [0]:
catalogos = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
assert CATALOGO in catalogos, f"El catálogo '{CATALOGO}' no existe. Disponibles: {catalogos}"

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}
    COMMENT 'Proyecto ETL: BiciMAD, calidad del aire y meteorología en Madrid (2022)'
""")
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}
    COMMENT 'Ficheros de origen tal cual llegan, una carpeta por fuente'
""")
spark.sql(f"USE {CATALOGO}.{ESQUEMA}")

In [0]:
for fuente, ruta in RUTAS.items():
    dbutils.fs.mkdirs(ruta)

In [0]:
print(f"Catálogo: {CATALOGO} | Esquema: {ESQUEMA} | Volumen: {RAW}")
display(dbutils.fs.ls(RAW))